# Gold Layer - Business Model (Full Load)

## Creating Dimension tables

### Table: gold_dim_customers
###### common steps
- rename columns
- Sort the columns into logical groups

In [ ]:
%%sql

CREATE OR REPLACE TABLE sales_lakehouse.dbo.gold_dim_customers
AS
SELECT
-- creating a unique key for dim_customer table
    ROW_NUMBER() OVER (ORDER BY ci.cst_id) AS customer_key, -- surrogate key
    ci.cst_id AS customer_id,
    ci.cst_key AS customer_number,
    ci.cst_firstname AS first_name,
    ci.cst_lastname AS last_name,
    la.cntry AS country,
    ci.cst_marital_status AS marital_status,

-- fixing data integrity issue
    CASE WHEN ci.cst_gndr != 'Unknown' THEN ci.cst_gndr
         ELSE COALESCE(cu.gen, 'Unknown')
    END AS gender,
    cu.bdate AS birthdate,
    ci.cst_create_date AS create_date
FROM sales_lakehouse.dbo.silver_crm_customer_info ci
LEFT JOIN sales_lakehouse.dbo.silver_erp_customers cu
ON ci.cst_key = cu.cid
LEFT JOIN sales_lakehouse.dbo.silver_erp_location la
ON ci.cst_key = la.cid;

### Table: gold_dim_products

In [ ]:
CREATE OR REPLACE TABLE sales_lakehouse.dbo.gold_dim_products
AS
SELECT
-- creating a unique key for dim_customer table
    ROW_NUMBER() OVER(ORDER BY prd_start_dt, prd_key) AS product_key, --surrogate key
    pr.prd_id AS product_id,
    pr.prd_key AS product_number,
    pr.prd_nm AS product_name,
    pr.cat_id AS category_id,
    pc.cat AS category,
    pc.subcat AS subcategory,
    pc.maintenance,
    pr.prd_cost AS cost,
    pr.prd_line AS product_line,
    pr.prd_start_dt AS start_date
FROM sales_lakehouse.dbo.silver_crm_product_info pr
LEFT JOIN sales_lakehouse.dbo.silver_erp_product_category pc
ON pr.cat_id = pc.id
WHERE pr.prd_end_dt IS NULL; -- Considering only current data and not the historical data


## Creating Fact tables

### Table: gold_fact_sales

In [ ]:
CREATE OR REPLACE TABLE sales_lakehouse.dbo.gold_fact_sales
AS
SELECT
    sls_ord_num AS order_number,
    cu.customer_key,
    pr.product_key,
    sd.sls_order_dt AS order_date,
    sd.sls_ship_dt AS shipping_date,
    sd.sls_due_dt AS due_date,
    sd.sls_sales AS sales_amount,
    sd.sls_quantity AS quantity,
    sd.sls_price AS price
FROM sales_lakehouse.dbo.silver_crm_sales_details sd
LEFT JOIN sales_lakehouse.dbo.gold_dim_customers cu      -- get the surrogate keys from the dim tables to connect facts with dims
ON sd.sls_cust_id = cu.customer_id
LEFT JOIN sales_lakehouse.dbo.gold_dim_products pr
ON sd.sls_prd_key = pr.product_number;
